# Lab 5: Working with Protein Families and Protein Structures

Goals
- Explore experimental (or predicted) protein structures; 
- Learn to use hmmer to work with protein families. 

Topics: 
- Fetch & parse PDB files (using PDB parser or not using)
- Explore structures (contact map)
- Visualize protein structuress (using Mol* in web browser)
- Use hmmer (command lines)

**Structure:** Ubiquitin `1UBQ` — 76 residues, α/β fold, well-studied, AlphaFold model available.

> **Cells marked `### YOUR CODE HERE ###`** are blanks for you to complete.  
> Read each docstring and the surrounding code carefully before filling in.

---
## Part 0 — Environment check and VS code server problem that you might have encountered. 

## VSCode problem?
We have been using VSCode and Silo server for a few weeks. By now, some of us may run into the problem that we can no longer remote login into Silo via VS Code as the server keeps rejecting that!

We will learn a simple way to fix the issue. Even if you don't have this issue yet, it will still be a good practice to try the following to see what's behind VS code server. 
- Remote login using ssh (e.g., in Mac terminal) -- yes, you can still login into silo even if you cannot do it via VS Code. 
- Check the .vscode-server folder under your home directory. You will see many files and folders under there. 
- BTW, in Linux, a file or directory whose name starts with a dot (.) is treated as hidden. The dot is simply a naming convention. Linux doesn't attach a special "hidden" attribute to these files. Dotfiles are commonly used for configuration and application data so they don't clutter your normal directory listing.
- Delete some of the large files/folders under .vscode-server. Alternatively, you can delete the entire .vscode-server folder!
- Then you shall be able to use VS Code to login into Silo server. 

Here are more explanations.
- When you use VS Code to login into a remote server like Silo, a VS Code server is set up there.  
- Remote .vscode-server takes up a lot of space (for installing the extensions etc).
- The reason that the server rejects VS Code remote connection is likely that the storage you used on the server is beyond your given quota. 
- You can delete the .vscode-server directory to free space. The next time you log in to this server, the system will automatically install another .vscode-server directory.

In [ ]:
from io import StringIO
import os, json, warnings, re
from collections import Counter

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import pandas as pd
import requests

from Bio.PDB import (
    PDBParser, PDBList, MMCIFParser,
    PPBuilder, NeighborSearch, is_aa
)
from Bio.SeqUtils import seq1

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'monospace',
})

print('All imports successful ✓')
import Bio; print(f'Biopython {Bio.__version__}  |  NumPy {np.__version__}')

---
## Part 1 — Fetching & parsing a PDB file

Every PDB entry has a 4-character **PDB ID**.  
BioPython's `PDBList` downloads the flat file; `PDBParser` parses it into the **SMCRA hierarchy**:  
Structure → Model → Chain → Residue → Atom.

Residue `id[0]` flags:
- `' '` — standard amino acid  
- `'H_XXX'` — HETATM ligand  
- `'W'` — water

In this section, you will learn how to fetch and parse a PDB file using PDB parser in the Bio module. 

In [ ]:
PDB_ID = '1ubq'
os.makedirs('data', exist_ok=True)

pdbl = PDBList(verbose=False)
path = pdbl.retrieve_pdb_file(PDB_ID, pdir='data', file_format='pdb')
print(f'Downloaded: {path}')

## Pause here -- check the PDB file and see it is downloaded correctly
Question: what is the size of the file?

Your answer: 

In [ ]:
parser    = PDBParser(QUIET=True)
structure = parser.get_structure(PDB_ID.upper(), path)
model     = next(structure.get_models())

print(f'Structure ID : {structure.id}')
print(f'Models       : {len(list(structure.get_models()))}')

rows = []
for chain in model:
    std_res   = [r for r in chain if r.id[0] == ' ']
    het_res   = [r for r in chain if r.id[0].startswith('H')]
    water_res = [r for r in chain if r.id[0] == 'W']
    atoms     = list(chain.get_atoms())
    rows.append({
        'Chain': chain.id,
        'Std residues': len(std_res),
        'HETATM groups': len(het_res),
        'Water molecules': len(water_res),
        'Total atoms': len(atoms),
    })
    print(f"std_res[0]: {std_res[0]}")

print('\nChain composition:')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ── PRACTICE 1.1 ────────────────────────────────────────────────────────────
# Print the residue number, 3-letter name, and number of atoms for
# the LAST 5 residues of chain A.
#
# Hint: get chain A with model['A'], filter for standard residues,
# then slice the last 5 with [-5:].
# For each residue r, len(list(r.get_atoms())) gives the atom count.
# ────────────────────────────────────────────────────────────────────────────
chain_A      = model['A']
std_residues = [r for r in chain_A if r.id[0] == ' ']

### YOUR CODE HERE ###

In [ ]:
ppb          = PPBuilder()
polypeptides = ppb.build_peptides(structure)
full_seq     = ''.join(str(pp.get_sequence()) for pp in polypeptides)

print(f'Sequence length: {len(full_seq)} residues')
print(full_seq)

In [ ]:
# ── PRACTICE 1.2 ────────────────────────────────────────────────────────────
# Count the frequency of each amino acid in full_seq and print the
# top 5 most common residues with their counts and percentages.
#
# Hint: collections.Counter is already imported.
# Percentage = count / len(full_seq) * 100
# ────────────────────────────────────────────────────────────────────────────

### YOUR CODE HERE ###


---
## Part 2 — Contact map

A **contact map** encodes which residue pairs are spatially close.  
Convention: two residues are *in contact* if their Cα–Cα distance < **8 Å**.  
Trivial sequential neighbours (|i − j| ≤ 2) are excluded.  

Patterns to look for:
- Diagonal streaks → α-helix
- Off-diagonal lines parallel to the main diagonal → parallel β-sheet
- Off-diagonal lines perpendicular to the main diagonal → antiparallel β-sheet



In [ ]:
# --- Practice 2.1 ----
# For this part, you will explore the most "ancient" way to work with PDB files: 
# reading lines from the files and parse the lines directly to extract the information that you need, 
# for example, the coordinates of the CA atoms of all the standard residues in a chain. 
# Implement the get_ca_atoms() function below. 

### COMPLETE CODE BELOW ###
def get_ca_atoms(file_path, chain_id):
    """
    Extracts all alpha carbon (CA) atoms from a specified chain in a PDB file.

    Parameters:
    file_path (str): Path to the PDB file. e.g. 'data/pdb1ubq.ent'
    chain_id (str): Chain identifier. e,g, 'A'

    Returns:
    list: List of CA atom coordinates, where each contains the coordinates of a CA atom.
    e.g., a typical list may look like this [[26.266, 25.413, 2.842], [26.85, 29.021, 3.898],...]
    """

    ### YOUR CODE HERE ###


## Run below cell only after you completed the get_ca_atoms() above

In [ ]:
coords_ca = get_ca_atoms('data/pdb1ubq.ent', 'A')
n = len(coords_ca)
# Full Cα pairwise distance matrix  (n × n)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist_matrix[i, j] = np.linalg.norm(np.array(coords_ca[i]) - np.array(coords_ca[j]))

print(f'Distance matrix shape: {dist_matrix.shape}')
print(f'Min non-self distance : {dist_matrix[dist_matrix > 0].min():.2f} Å')
print(f'Max distance          : {dist_matrix.max():.2f} Å')

In [ ]:
# ── PRACTICE 2.2 ────────────────────────────────────────────────────────────
# Build the binary contact map.
#
# (a) Create a boolean array `contact_map` where True means
#     dist_matrix[i,j] < 8.0 Å.
# (b) Set the diagonal to False (no self-contacts).
#     Hint: np.fill_diagonal(contact_map, False)
# (c) Build `long_range` — contacts where |i − j| > 2.
#     Hint: np.abs(np.subtract.outer(np.arange(n), np.arange(n))) > 2
# (d) Print the total number of long-range contacts (remember: each
#     pair is counted twice in a symmetric matrix, so divide by 2).
# ────────────────────────────────────────────────────────────────────────────

CONTACT_THRESHOLD = 8.0

### YOUR CODE HERE ###

In [ ]:
# Run this cell after completing Practice 2.2
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(dist_matrix, cmap='viridis_r', origin='lower', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Cα–Cα distance (Å)', shrink=0.85)
axes[0].set_title('Cα distance matrix', fontsize=10)
axes[0].set_xlabel('Residue index'); axes[0].set_ylabel('Residue index')

axes[1].imshow(long_range, cmap='Blues', origin='lower', aspect='auto', vmin=0, vmax=1)
axes[1].set_title(f'Contact map  (d < {CONTACT_THRESHOLD} Å, |i−j| > 2)', fontsize=10)
axes[1].set_xlabel('Residue index'); axes[1].set_ylabel('Residue index')

# Known β-strand positions for 1UBQ
beta_regions = [(0,6),(10,16),(39,44),(47,49),(63,71)]
helix        = (22, 33)
for ax in axes:
    for (s, e) in beta_regions:
        ax.axvspan(s, e, alpha=0.12, color='#457b9d')
        ax.axhspan(s, e, alpha=0.12, color='#457b9d')
    ax.axvspan(*helix, alpha=0.12, color='#e76f51')
    ax.axhspan(*helix, alpha=0.12, color='#e76f51')

plt.tight_layout()
plt.savefig('data/contact_map.png', dpi=130, bbox_inches='tight')
plt.show()

## Part 3. 3D Molecule Viewer. 
Over the years, different visualiation tools have been developed for 3D structure visualization. 

Mol* (pronounced "Molstar") is such a tool. It is a web-based and open-source 3D molecular viewer used to display and analyze large-scale biological macromolecule data. 

For this practice, we will be using Mol*. 
- Go to [Mol* viewer](https://molstar.org/viewer/)
- Explore the viewer using the default PDB (1TQN)
- What to explore? 
 - Presentations (Cartoon, Spacefill, Surface)
 - Residue selection (e.g., select an imortant residue and show it in a different color to highlight)
 - Output images
- Uploaded your own pdb (here: 1ubq you just downloaded) and explore the structure similarly. 

Answer the following questions:

- Is 1TQN mainly α, β, α/β, or α+β?
- How about 1UBQ?

## Your answers

- Is 1TQN mainly α, β, α/β, or α+β?
- How about 1UBQ?


## Part VI: working with protein families (for annotation etc) using HMMER

HMMER is a suite of programs for biological sequence analysis using profile hidden Markov models (profile HMMs). Unlike BLAST, which is for searching sequences against sequences, HMMER can be used for more sensitive searching by leveraging profile HMM of protein families. 

Core programs in the HMMER suite:
- hmmbuild	MSA → profile HMM	Build a profile HMM from a curated alignment
- hmmsearch	profile HMM vs sequence DB	Find all members of a family in a database
- hmmscan	sequence vs HMM DB (Pfam)	Annotate domains in a query protein
- phmmer	single sequence vs sequence DB	Protein-protein BLAST equivalent
- jackhmmer	iterative single sequence	PSI-BLAST equivalent, iterative enrichment
- hmmpress	HMM database	Compress & index HMM library for fast hmmscan
- hmmfetch	HMM database	Retrieve individual HMMs from a library

All the exercises for this part will be done using command lines.

For this practice, we will use a small database of protein families (in HMMs) and annoate query proteins by searching them against the family database. 

If you have active Bio2 environment created for this class, HMMER is ready for you to use.  

NOTE: You need to activate Bio2 environment separately in terminal so you can use command lines.

See [HMMER tutorial](http://hmmer.org/documentation.html)

Databases available under /nobackup/yye/Bio2/datasets/hmmer (on silo):
- RVT-All.hmm -- a database of HMMs of RTs (reverse transcriptases)
- q1_protein.faa -- a file with 5 proteins 

Copy these two files to your own folder (you cannot write to the /nobackup/yye/..). Then do the following.

- Step 1: compress the hmm file using hmmpress; command: hmmpress RVT-All.hmm

- Step 2: search query against the HMM database using hmmscan; command: hmmscan --cpu 2 -E 0.001 --domE 0.001 -o q1_vs_RVT.txt --tblout q1_vs_RVT.tbl --domtblout q1_vs_RVT.domtbl RVT-All.hmm q1_protein.faa

Some notes:
- Make sure that you inspect the files created by hmmpress before you dive into the next step.  
- You can look up usage online for each of the commands; you may also simply call the command without providing any arguments to show the usage. 

Answer the questions below. 

4.1 Based on your results, does q1_protein.faa contain proteins that are putative RT?